In [1]:
import sys, os
# Get absolute path to the project root
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../.."))
sys.path.append(PROJECT_ROOT)

sys.path.append("/cluster/home/herminea/mental_health_project/workspace/utils")
sys.path.append("/cluster/home/herminea/mental_health_project/workspace/scripts")

In [2]:
import pickle
import numpy as np
from utils.functional_connectivity.fc_compute import (
    compute_fc_per_mode,
    compute_fc_whole_band
)
from utils.io.io_data import load_roi_timeseries_runs

# --- Path to ONE decomposition file ---
src_path = "/cluster/home/herminea/mental_health_project/workspace/results/fmri_prep/mvmd/imfs/NDAR_INVAL101MH2_run1.pkl"   # <-- change this

with open(src_path, "rb") as f:
    data = pickle.load(f)

subj = data["subject"]
imfs = data["imfs"]
freqs = np.array(data["freqs"])
run_file = data["run_file"]

print("Subject:", subj)
print("IMFs shape:", imfs.shape)

Subject: NDAR_INVAL101MH2
IMFs shape: (8, 488, 434)


In [3]:
low, high = 0.005, 0.25
valid_idx = np.where((freqs >= low) & (freqs <= high))[0]

imfs = imfs[valid_idx]
freqs = freqs[valid_idx]

print("Filtered IMFs shape:", imfs.shape)

Filtered IMFs shape: (6, 488, 434)


In [4]:
fc_modes = compute_fc_per_mode(imfs)
print("FC modes shape:", fc_modes.shape)

FC modes shape: (6, 434, 434)


In [10]:
import os
import pickle

fc_dir = "/cluster/home/herminea/mental_health_project/workspace/results/fmri_prep/vlmd/fc"

# pick one file
fname = next(f for f in os.listdir(fc_dir) if f.endswith("_fc.pkl"))
path = os.path.join(fc_dir, fname)

print("Checking file:", fname)

with open(path, "rb") as f:
    data = pickle.load(f)

print("fc_modes shape:", data["fc_modes"].shape)

if data["fc_whole"] is None:
    print("fc_whole is None")
else:
    print("fc_whole shape:", data["fc_whole"].shape)

Checking file: NDAR_INVCW125BLA_run4_fc.pkl
fc_modes shape: (6, 434, 434)
fc_whole shape: (434, 434)


In [11]:
from utils.io.io_data import load_roi_timeseries_runs

h5_path = "/cluster/home/herminea/mental_health_project/workspace/data/roi_timeseries/sub-NDARINVAL101MH2_roi_timeseries.h5"

runs = load_roi_timeseries_runs(h5_path)

print("Available runs:", list(runs.keys())[:5])
first_run = list(runs.keys())[0]
X = runs[first_run]
print("Loaded X shape:", X.shape)
print("Example run name:", first_run)

Available runs: ['sub-NDARINVAL101MH2_task-restAP_run-01_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold', 'sub-NDARINVAL101MH2_task-restAP_run-02_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold', 'sub-NDARINVAL101MH2_task-restPA_run-01_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold', 'sub-NDARINVAL101MH2_task-restPA_run-02_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold']
Loaded X shape: (434, 488)
Example run name: sub-NDARINVAL101MH2_task-restAP_run-01_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold


In [12]:
import os, pickle, numpy as np
from utils.io.io_data import load_roi_timeseries_runs
from scipy.stats import zscore
from scipy.signal import butter, filtfilt

# --- pick one run result ---
fc_pkl = "/cluster/home/herminea/mental_health_project/workspace/results/fmri_prep/vlmd/fc/NDAR_INVCW125BLA_run4_fc.pkl"

with open(fc_pkl, "rb") as f:
    d = pickle.load(f)

# --- load raw ROI×T time series ---
run_file = d["run_file"].replace("/test/", "/workspace/")  # adjust if needed
runs = load_roi_timeseries_runs(run_file)
X = runs[d["run_name"]]  # X is (R,T)
print("X shape:", X.shape)

tr = 0.8
fs = 1.0 / tr
lowcut, highcut = 0.01, 0.1

def butter_bandpass(data_RT, fs, lowcut, highcut, order=4, time_axis=1):
    nyq = 0.5 * fs
    low, high = lowcut / nyq, highcut / nyq
    b, a = butter(order, [low, high], btype="band")
    return filtfilt(b, a, data_RT, axis=time_axis)

# --- Correct: filter along time axis (axis=1 for R×T) ---
X_filt_time = butter_bandpass(X, fs, lowcut, highcut, time_axis=1)
X_filt_time = zscore(X_filt_time, axis=1)               # ROI-wise across time
fc_time = np.corrcoef(X_filt_time)                      # R×R
fc_time_z = np.arctanh(np.clip(fc_time, -0.999999, 0.999999))

# --- Wrong: filter along ROI axis (axis=0) ---
X_filt_roi = butter_bandpass(X, fs, lowcut, highcut, time_axis=0)
X_filt_roi = zscore(X_filt_roi, axis=1)
fc_roi = np.corrcoef(X_filt_roi)
fc_roi_z = np.arctanh(np.clip(fc_roi, -0.999999, 0.999999))

# --- Compare to your saved fc_whole ---
fc_saved_z = d["fc_whole"]

# Similarity metrics
def mat_corr(A, B):
    iu = np.triu_indices_from(A, k=1)
    a = A[iu].ravel()
    b = B[iu].ravel()
    return np.corrcoef(a, b)[0,1]

print("corr(saved, time-axis correct):", mat_corr(fc_saved_z, fc_time_z))
print("corr(saved, ROI-axis wrong):   ", mat_corr(fc_saved_z, fc_roi_z))
print("max|saved-time|:", np.max(np.abs(fc_saved_z - fc_time_z)))
print("max|saved-roi|: ", np.max(np.abs(fc_saved_z - fc_roi_z)))

X shape: (434, 488)
corr(saved, time-axis correct): 1.0
corr(saved, ROI-axis wrong):    0.18976267635462116
max|saved-time|: 4.884981308350689e-15
max|saved-roi|:  2.7164886061389524
